In [ ]:
# 02 Preprocessing validation
#
# Question. Does the input pipeline produce the documented tensors (crop, [-1, 1] range, channels, dtype, shape)?
#
# Type. Validation of conventions in docs/preprocessing-and-weights.md. Synthetic checks run always; a real-image check runs only if USE_REAL_IMAGE = True.
#
# Note: the 2x-1 convention is a frozen, labelled assumption for the TensorFlow .h5 weights (see docs/decisions/0002-batchnorm-and-preprocessing.md).
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ct_iqa.config.schema import ModelConfig
from ct_iqa.models.registry import build_model
from ct_iqa.preprocessing.transforms import preprocess_ct_image


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# 1. Synthetic ramp image [SAFE]
# No data is read.
x = np.tile(np.linspace(0, 1, 512, dtype=np.float32), (512, 1))
y = preprocess_ct_image(x, 224)
checks = {
    "shape is 224x224": y.shape == (224, 224),
    "dtype is float32": y.dtype == np.float32,
    "centre crop (columns 144..367)": bool(np.isclose(y.min(), 2 * x[0, 144] - 1) and np.isclose(y.max(), 2 * x[0, 367] - 1)),
}
show("Preprocessing checks on a synthetic ramp", pd.DataFrame({"passed": checks}))
show("Value range before and after crop and 2x-1", pd.DataFrame({"min": [x.min(), y.min()], "max": [x.max(), y.max()]}, index=["input", "after preprocessing"]).round(4))

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(x[0], label="input row ([0, 1])")
ax.plot(np.arange(144, 368), y[0], label="after crop and 2x-1 ([-1, 1])")
ax.set(xlabel="column", title="Ramp profile"); ax.legend(); ax.grid(alpha=0.3)
plt.show()


In [ ]:
# 2. Model input and BatchNorm epsilon [SAFE]
# Builds the architecture without pretrained weights to read its BatchNorm epsilon; no forward pass.
model, _ = build_model(ModelConfig(name="resnet50_regression", dropout_p=0.5, bn_eps=1.001e-5))
eps = {m.eps for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)}
t = torch.from_numpy(y).unsqueeze(0).unsqueeze(0)
show("Model input and BatchNorm epsilon", pd.DataFrame({"value": {"distinct BatchNorm eps values": eps, "input tensor shape": tuple(t.shape),
                                "channels used by the backbone": "1 -> replicated to 3 inside forward"}}))


In [ ]:
# 3. Optional: one real image [OPT-IN: READS IMAGES]
# Reads one training image only when USE_REAL_IMAGE = True; off by default.
USE_REAL_IMAGE = False
if USE_REAL_IMAGE:
    from PIL import Image

    p = sorted(Path("data/raw/ldct_iqac/train/image").glob("*.tif"))[0]
    with Image.open(p) as im:
        mode, size, arr = im.mode, im.size, np.array(im, dtype=np.float32)
    out = preprocess_ct_image(arr, 224)
    show("Real image check", pd.DataFrame({"value": {"file": p.name, "mode": mode, "size": size, "raw range": (float(arr.min()), float(arr.max())),
                                    "processed range": (float(out.min()), float(out.max())), "processed shape": out.shape}}))
